# Lab competition — starter

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/convexpi/missions/blob/main/starters/lab_starter.ipynb)

This is the starter for the **Lab** competitions (e.g. *The Open Leaderboard*). The competition is
graded on a **hidden synthetic market** — you never see the exact data you're scored on. But the
market's *structure* is public, so you develop against your **own** instances of the same market,
fit a strategy on the **in-sample** period, and submit the strategy code. The grader then runs it on
the hidden **out-of-sample** holdout.

> The whole point: a high in-sample score is trivial (just overfit). You're ranked on **OOS Sharpe** —
> the part you can't see. This notebook shows how to get the in-sample data, explore it, fit a
> strategy, and check it OOS *yourself* before submitting.

## Setup

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q "git+https://github.com/convexpi/lab.git@6c69a32ccc8b46bd3c9804866131c74b85ee3d7c"

In [ ]:
import numpy as np
from convexpi.lab import SyntheticMarket, Backtest, Strategy
print('Ready.')

## 1. Get the in-sample data

`SyntheticMarket` *is* the data. The grader builds one with a **private seed**; you build your own
with any seed. Same structure (the planted alphas live in the same named features), different draw —
so a strategy that works across your instances should generalize to the grader's.

In [ ]:
market = SyntheticMarket(n_assets=200, n_days=1260, seed=0)

prices   = market.prices('train')      # (T_train, N) in-sample daily closes — the data you fit on
features = market.features('train')    # dict: feature_name -> (T_train, N) cross-section each day

print('features    :', market.FEATURE_NAMES)
print('in-sample   :', prices.shape[0], 'days x', prices.shape[1], 'stocks')
print('holdout     :', market.prices('test').shape[0], 'days (you never see the grader\'s version)')

## 2. Explore — which features actually predict?

Most features are pure noise; one or two are **planted alphas**. Measure each feature's information
coefficient (IC) — the daily cross-sectional correlation between the feature and next-day returns —
on the in-sample data. This is your *discovery* step.

In [ ]:
rets = prices[1:] / prices[:-1] - 1            # next-day returns, (T-1, N)
print(f"{'feature':12} {'mean IC':>9}")
for name in market.FEATURE_NAMES:
    f = features[name][:-1]                     # align with next-day returns
    ics = [np.corrcoef(f[t], rets[t])[0, 1] for t in range(len(rets)) if np.std(f[t]) > 0]
    print(f"{name:12} {np.nanmean(ics):>9.4f}")

The features with a non-trivial mean IC are your signal; the `noise_*` ones should hover near zero.
Build a strategy that leans on the real signals.

## 3. Fit a strategy

A submission is a class **`MyStrategy(Strategy)`** implementing `on_day(...) -> target weights`.
Keep it simple and robust — overfitting the in-sample draw will just hurt you OOS.

In [ ]:
class MyStrategy(Strategy):
    """Long/short on a small combination of the signals you trust."""
    weights = {'mom_1m': 1.0, 'val_bm': 0.5}   # <- set these from your IC exploration above
    frac = 0.5                                   # fraction of names in each leg

    def on_day(self, day, features, prices, portfolio):
        n = len(prices)
        score = np.zeros(n)
        for name, wt in self.weights.items():
            score += wt * np.nan_to_num(features.get(name, np.zeros(n)))
        k = max(1, int(n * self.frac))
        order = np.argsort(score)
        w = np.zeros(n)
        w[order[-k:]] =  1.0 / k                 # long the strongest
        w[order[:k]]  = -1.0 / k                 # short the weakest
        return w

## 4. Score it yourself — in-sample vs out-of-sample

Run the **same backtest the grader uses** (weekly rebalance, transaction costs) on both splits.
A high IS Sharpe with a much lower OOS Sharpe means you overfit. Aim for a strategy whose OOS Sharpe
holds up — that's what the leaderboard ranks.

In [ ]:
bt = Backtest(tc_bps=10, rebalance_every=5)
strat = MyStrategy()
is_res  = bt.run(strat, market.prices('train'), market.features('train'))
oos_res = bt.run(strat, market.prices('test'),  market.features('test'))

print(f"In-sample  Sharpe : {is_res.sharpe:6.3f}")
print(f"Out-of-sample Sharpe: {oos_res.sharpe:6.3f}")
ratio = oos_res.sharpe / is_res.sharpe if is_res.sharpe > 0 else 0
print(f"Overfitting ratio  : {ratio:6.2f}   (1.0 = held up perfectly; near 0 = overfit)")

Test across **several seeds** (re-run with `seed=1, 2, 3, …`) — a strategy that's only good on one
draw won't survive the hidden market.

## 5. Submit

Copy your `MyStrategy` class (the cell in step 3) into the competition's **Submit** page. The grader
runs it on the hidden out-of-sample holdout and posts your OOS Sharpe to the leaderboard. Iterate.